In [24]:
import pandas as pd
import plotly.express as px

# Roll A (andmete laadimine)
# RAW andmete laadimine failist ja nendest DataFrame loomine
df_sales = pd.read_csv("sales.csv")
df_customers = pd.read_csv("customers.csv")

# Sales tabeli shape ja head 
print("SALES kuju:")
print(df_sales.shape)

print("\nSALES esimesed read:")
display(df_sales.head())

# Customers tabeli shape ja head
print("\nCUSTOMERS kuju:")  
print(df_customers.shape)
print("\nCUSTOMERS esimesed read:")
display(df_customers.head())

# Andmete ühendamine customer_id alusel
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

# Liidetud DataFrame'i shape, dtypes ja head()
print("MERGED DataFrame kuju:")
print(df.shape)

print("\nMERGED DataFrame andmetüübid:")
print(df.dtypes)

print("\nMERGED DataFrame esimesed read:")
display(df.head())

# Roll B (Andmete puhastamine)

# Duplikaatide kontroll ja eemaldamine
duplicate_count = df.duplicated(subset=["invoice_id"]).sum()

print("Duplikaate invoice_id järgi:", duplicate_count)

df = df.drop_duplicates(
    subset=["invoice_id"],
    keep="first"
)

print("Shape pärast duplikaatide eemaldamist:", df.shape)

# NULL väärtuste kontroll ja eemaldamine
df = df.dropna(
    subset=["customer_id", "sale_date", "total_price"]
)

print("Shape pärast NULL-ide eemaldamist:", df.shape)

# Kuupäevade teisendamine datetime formaati ja kuupäevaline piiramine
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
df = df[df["sale_date"] < "2025-03-01"]

# Outliers kontroll ja eemaldamine
negative_prices = (df["total_price"] <= 0).sum()

print("Null või negatiivse total_price väärtusega ridu:",
      negative_prices)
df = df[df["total_price"] > 0].copy()

print("Shape pärast outlier'ite eemaldamist:", df.shape)

# Puhastusraport
print("=== PUHASTUSRAPORT ===")
print(f"\nLõplik shape: {df.shape}")
print(f"Unikaalseid kliente: {df['customer_id'].nunique()}")

print(
    "Kuupäevavahemik:",
    df["sale_date"].min().date(),
    "kuni",
    df["sale_date"].max().date()
)

# Roll C (RFM kliendisegmenteerimine)

# Viitekuupäeva seadmine
today = pd.to_datetime("2025-02-28")
# Regency arvutus
recency = (
    df.groupby("customer_id")["sale_date"]
      .max()
      .reset_index()
)

recency["recency_days"] = (
    today - recency["sale_date"]
).dt.days

display(recency.head())
# Frequency arvutus
frequency = (
    df.groupby("customer_id")["invoice_id"]
      .nunique()
      .reset_index()
      .rename(columns={"invoice_id": "frequency"})
)
display(frequency.head())
# Monetary arvutus
monetary = (
    df.groupby("customer_id")["total_price"]
      .sum()
      .reset_index()
      .rename(columns={"total_price": "monetary"})
)
display(monetary.head())

recency = recency[
    ["customer_id", "recency_days"]
]
# RFM tabeli loomine

rfm = pd.merge(
    recency,
    frequency,
    on="customer_id"
)

rfm = pd.merge(
    rfm,
    monetary,
    on="customer_id"
)

display(rfm.head())
# RFM skooride arvutamine
rfm["R_score"] = pd.qcut(
    rfm["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1]
)
rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
)
# RFM skooride ühendamine ja segmenteerimine
rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)
rfm["RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + rfm["M_score"]
)
def assign_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 11:
        return "Loyal Customers"
    elif score >= 9:
        return "Regular Customers"
    elif score >= 7:
        return "New Customers"
    elif score >= 5:
        return "At Risk"
    else:
        return "Lost"

rfm["segment"] = rfm["RFM_Score"].apply(assign_segment)
actions = {
    "VIP Champions": "Early access, VIP sooduskoodid",
    "Loyal Customers": "Lojaalsusprogramm, preemiad",
    "Regular Customers": "Cross-sell kampaaniad",
    "New Customers": "Onboarding, welcome-sari",
    "At Risk": "Win-back kampaania, personaliseeritud e-mail",
    "Lost": "Viimane taasaktiveerimise pakkumine"
}

rfm["Action"] = rfm["segment"].map(actions)
# Segmentide kokkuvõte
segment_summary = (
    rfm.groupby("segment")
       .size()
       .reset_index(name="customers")
)
segment_summary["share_pct"] = (
    segment_summary["customers"]
    / segment_summary["customers"].sum()
    * 100
).round(1)
segment_order = [
    "VIP Champions",
    "Loyal Customers",
    "Regular Customers",
    "New Customers",
    "At Risk",
    "Lost"
]

segment_summary["segment"] = pd.Categorical(
    segment_summary["segment"],
    categories=segment_order,
    ordered=True
)

segment_summary = segment_summary.sort_values("segment")
display(segment_summary)

# Roll D (Visualiseerimine ja leiud)
# Ees- ja perenimede lisamine RFM tabelisse
import plotly.express as px

# Kliendinimed RFM tabelisse
customer_names = (
    df[["customer_id", "first_name", "last_name"]]
    .drop_duplicates(subset=["customer_id"])
)

rfm = rfm.merge(
    customer_names,
    on="customer_id",
    how="left"
)

rfm["client_name"] = (
    rfm["first_name"].fillna("") + " " +
    rfm["last_name"].fillna("")
).str.strip()

# Kui nimi puudub, kasutame customer_id-d
rfm["client_name"] = rfm["client_name"].mask(
    rfm["client_name"] == "",
    "Klient " + rfm["customer_id"].astype(str)
)
# Diagramm 1 — tulpdiagramm: segmentide klientide arv
# Diagrammi tabel
segment_counts = (
    rfm.groupby("segment")
       .size()
       .reset_index(name="Klientide arv")
)

segment_order = [
    "VIP Champions",
    "Loyal Customers",
    "Regular Customers",
    "New Customers",
    "At Risk",
    "Lost"
]

segment_counts["segment"] = pd.Categorical(
    segment_counts["segment"],
    categories=segment_order,
    ordered=True
)

segment_counts = segment_counts.sort_values("segment")
# Diagramm 1 ise
fig1 = px.bar(
    segment_counts,
    x="segment",
    y="Klientide arv",
    text="Klientide arv",
    title="UrbanStyle kliendisegmentide jaotus",
    labels={
        "segment": "Kliendisegment",
        "Klientide arv": "Klientide arv"
    }
)

fig1.update_traces(
    textposition="outside"
)

fig1.update_layout(
    showlegend=False,
    xaxis_title=None,
    yaxis_title="Klientide arv",
    plot_bgcolor="white"
)

fig1.update_yaxes(
    showgrid=True,
    gridcolor="lightgrey"
)

fig1.show()

# Diagramm 2 hajuvusdiagramm (scatter): recency_days vs monetary_value
fig2 = px.scatter(
    rfm,
    x="recency_days",
    y="monetary",
    color="segment",
    size="frequency",
    hover_name="client_name",
    hover_data={
        "frequency": True,
        "RFM_Score": True,
        "customer_id": False
    },
    title="UrbanStyle kliendisegmendid RFM analüüsi põhjal",
    labels={
        "recency_days": "Päevi viimasest ostust",
        "monetary": "Kogukulutus (€)",
        "frequency": "Ostude arv",
        "segment": "Segment",
        "RFM_Score": "RFM skoor"
    }
)

fig2.update_layout(
    plot_bgcolor="white"
)

fig2.update_xaxes(
    showgrid=True,
    gridcolor="lightgrey"
)

fig2.update_yaxes(
    showgrid=True,
    gridcolor="lightgrey"
)

fig2.show()

# Diagramm 3 — top 10 VIP klienti kogukulutuse järgi
# Top 10 VIP klientide tabel
top_vip = (
    rfm[
        rfm["segment"] == "VIP Champions"
    ]
    .nlargest(10, "monetary")
    .sort_values("monetary")
)
# tulpdiagramm
fig3 = px.bar(
    top_vip,
    x="monetary",
    y="client_name",
    orientation="h",
    text="monetary",
    title="Top 10 VIP klienti kogukulutuse järgi",
    labels={
        "monetary": "Kogukulutus (€)",
        "client_name": "Klient"
    }
)

fig3.update_traces(
    texttemplate="%{text:.0f} €",
    textposition="outside"
)

fig3.update_layout(
    showlegend=False,
    xaxis_title="Kogukulutus (€)",
    yaxis_title=None,
    plot_bgcolor="white"
)

fig3.update_xaxes(
    showgrid=True,
    gridcolor="lightgrey"
)

fig3.show()
# Äritõlgendus Markole
# VIP klientide arv
vip_count = (
    rfm["segment"] == "VIP Champions"
).sum()
# AT Risk klientide arv
at_risk_count = (
    rfm["segment"] == "At Risk"
).sum()
# VIP klientide kogukulutused
vip_revenue = rfm.loc[
    rfm["segment"] == "VIP Champions",
    "monetary"
].sum()
# Kõikide klientide kogukulutused
total_revenue = rfm["monetary"].sum()
# VIP klientide osakaal kogukulutustes
vip_revenue_share = (vip_revenue / total_revenue) * 100
# Automaatne kokkuvõte Markole
print(
    f"UrbanStyle'i kliendibaasis on {vip_count} VIP Champions klienti, "
    f"kes moodustavad {vip_revenue_share:.1f}% klientide kogukulutusest. "
    f"Need kliendid on ettevõtte kõige väärtuslikum segment, kuna nad "
    f"ostavad sageli, on teinud oste hiljuti ning nende kogukulutus on kõrge. "
    f"Samal ajal kuulub {at_risk_count} klienti At Risk segmenti, mis näitab, "
    f"et osa klientidest on eemaldumas. Ettevõttel tasub keskenduda VIP "
    f"klientide hoidmisele ning At Risk klientide taasaktiveerimisele."
)

# CSV faili salvestamine
rfm.to_csv("rfm_segments.csv", index=False, encoding="utf-8-sig")

SALES kuju:
(15234, 11)

SALES esimesed read:


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart



CUSTOMERS kuju:
(3150, 9)

CUSTOMERS esimesed read:


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


MERGED DataFrame kuju:
(15234, 19)

MERGED DataFrame andmetüübid:
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object

MERGED DataFrame esimesed read:


...

,customer_id,frequency
0,2001.0,2
1,2004.0,2
2,2005.0,4
3,2006.0,1
4,2007.0,1


,customer_id,monetary
0,2001.0,203.92
1,2004.0,1198.56
2,2005.0,959.60
3,2006.0,327.06
4,2007.0,318.63


,customer_id,recency_days,frequency,monetary
0,2001.0,91,2,203.92
1,2004.0,71,2,1198.56
2,2005.0,148,4,959.60
3,2006.0,477,1,327.06
4,2007.0,29,1,318.63


,segment,customers,share_pct
5,VIP Champions,451,17.9
2,Loyal Customers,413,16.3
4,Regular Customers,529,20.9
3,New Customers,501,19.8
0,At Risk,370,14.6
1,Lost,262,10.4


UrbanStyle'i kliendibaasis on 451 VIP Champions klienti, kes moodustavad 42.8% klientide kogukulutusest. Need kliendid on ettevõtte kõige väärtuslikum segment, kuna nad ostavad sageli, on teinud oste hiljuti ning nende kogukulutus on kõrge. Samal ajal kuulub 370 klienti At Risk segmenti, mis näitab, et osa klientidest on eemaldumas. Ettevõttel tasub keskenduda VIP klientide hoidmisele ning At Risk klientide taasaktiveerimisele.


1. VIP programm. Pakkuda VIP Champions klientidele personaalseid soodustusi, varajast ligipääsu uutele kollektsioonidele ja lojaalsushüvesid, et suurendada nende püsimist ja ostusagedust.
2. Win-back kampaania. Suunata At Risk klientidele personaalne taasaktiveerimiskampaania, näiteks piiratud tähtajaga soodustus või soovitus varasemate ostude põhjal.
3. Nurture programm. Potential segmenti tasub järjepidevalt aktiveerida ristmüügi, komplektpakkumiste ja personaliseeritud soovitustega, et aidata neil liikuda Loyal ja VIP segmentidesse.